In [ ]:
# YASSER HOUSSEIN HASSAN
# PROJET

# Install cplex
!pip install cplex

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.0/57.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 MB 11.4 MB/s eta 0:00:00


In [39]:
from cplex import Cplex
from cplex.exceptions import CplexError

# Define the linear programming program
# Create an instance of Cplex

cpx=Cplex()
costs=[[0,75,115,20,35,70,70,35],[75,0,115,35,100,150,70,50],[115,115,0,120,120,75,55,90],[20,35,120,0,45,95,70,45],
       [35,100,120,45,0,60,95,60],[70,150,75,95,60,0,100,100],[70,70,55,70,95,100,0,40],[35,50,90,45,60,100,40,0]]
D=[23,15,16,8,17,28,12]
ntown=len(D)+1
truck=3
Q=50
var_x = [f"x_{i}_{j}_{k}" for i in range(ntown) for j in range(ntown) for k in range(truck) if i!=j]
var_y = [f"y_{i}_{k}" for i in range(ntown) for k in range(truck)]
var_u = [f"u_{i}_{k}" for i in range(ntown-1) for k in range(truck)]

# Add variables

cpx.variables.add(types=['B'] * len(var_x),
                          names=var_x)
cpx.variables.add(types=['B'] * len(var_y),
                          names=var_y)
cpx.variables.add(types=['C'] * len(var_u), lb=[1]*len(var_u), ub=[ntown-1]*len(var_u),
                          names=var_u)

constraints=[]
senses=[]
rhs=[]

# Constraints 1

for i in range(ntown-1):
        indices = [f"y_{i}_{k}" for k in range(truck)]
        cpx.linear_constraints.add(
            lin_expr=[[indices, [1] * len(indices)]],
            senses=['E'],
            rhs=[1]
        )

for j in range(ntown-1):
        indices = [f"x_{i}_{j}_{k}" for k in range(truck) for i in range(ntown) if i!=j]
        cpx.linear_constraints.add(
            lin_expr=[[indices, [1] * len(indices)]],
            senses=['E'],
            rhs=[1]
        )

# Constraints 2

for k in range(truck):
        indices = [f"y_{i}_{k}" for i in range(ntown-1)]
        cpx.linear_constraints.add(
            lin_expr=[[indices, D]],
            senses=['L'],
            rhs=[Q]
        )

# Constraints 3

for k in range(truck):
        for j in range(ntown):
            inflow = [f"x_{i}_{j}_{k}" for i in range(ntown) if i != j]
            outflow = [f"x_{j}_{i}_{k}" for i in range(ntown) if i != j]
            cpx.linear_constraints.add(
                lin_expr=[[inflow + outflow, [1] * len(inflow) + [-1] * len(outflow)]],
                senses=['E'],
                rhs=[0]
            )

 # Constraints 4

for k in range(truck):
   flow = [f"x_{i}_{ntown-1}_{k}" for i in range(ntown-1)]
   cpx.linear_constraints.add(
                lin_expr=[[flow, [1] * len(flow)]],
                senses=['E'],
                rhs=[1]
            )
for k in range(truck):
   flow2 = [f"x_{ntown-1}_{j}_{k}" for j in range(ntown-1)]
   cpx.linear_constraints.add(
                lin_expr=[[flow2, [1] * len(flow2)]],
                senses=['E'],
                rhs=[1]
            )

 # Constraints 5

for k in range(truck):
        for j in range(ntown):
            inflow = [f"x_{i}_{j}_{k}" for i in range(1,ntown) if i != j]
            outflow = [f"y_{j}_{k}"]
            cpx.linear_constraints.add(
            lin_expr=[[inflow + outflow, [1] * len(inflow) + [-1]]],
                senses=['E'],
                rhs=[0]
            )

# Constraints MTZ

for k in range(truck):
    for i in range(ntown-1):
        for j in range(ntown-1):
            if i != j:
                un=[f"u_{i}_{k}"]
                de=[f"u_{j}_{k}"]
                tr=[f"x_{i}_{j}_{k}"]
                cpx.linear_constraints.add(
                    lin_expr=[[un+de+tr, [1]*len(un)+[-1]*len(de)+[ntown]*len(tr)]],
                    senses=['L'],
                    rhs=[ntown-1]
                )

# Add constraints to the model

cpx.linear_constraints.add(lin_expr=constraints,senses=senses,rhs=rhs)

# Objective

cpx.objective.set_sense(cpx.objective.sense.minimize)
cpx.objective.set_linear(zip(var_x,[costs[i][j] for i in range(ntown) for j in range(ntown) for k in range(truck) if i!=j]))


# Solve the model

try:
    cpx.solve()
    sol_status=cpx.solution.get_status_string()
    opti_cost=cpx.solution.get_objective_value()
    print(f"Solution status: {sol_status}")
    print(f"Optimal cost: {opti_cost:.2f} €")
    solution_values = cpx.solution.get_values()
    for i, var_name in enumerate(var_x):
        if solution_values[i] == 1:
            print(f"{var_name}: {solution_values[i]}")
except CplexError as exc:
    print(exc)

Version identifier: 22.1.1.0 | 2023-06-15 | d64d5bd77
CPXPARAM_Read_DataCheck                          1
Tried aggregator 2 times.
MIP Presolve modified 294 coefficients.
Aggregator did 5 substitutions.
Reduced MIP has 189 rows, 208 columns, and 1069 nonzeros.
Reduced MIP has 187 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (1.02 ticks)
Found incumbent of value 660.000000 after 0.02 sec. (1.91 ticks)
Probing fixed 20 vars, tightened 0 bounds.
Probing time = 0.01 sec. (0.95 ticks)
Tried aggregator 2 times.
MIP Presolve eliminated 20 rows and 23 columns.
MIP Presolve modified 32 coefficients.
Aggregator did 3 substitutions.
Reduced MIP has 166 rows, 182 columns, and 925 nonzeros.
Reduced MIP has 164 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.01 sec. (0.72 ticks)
Probing time = 0.00 sec. (0.75 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 166 rows, 182 columns, and 925 nonzeros.
Reduced MIP has 164 binaries, 0 g

In [33]:
import numpy as np
import folium
from geopy.geocoders import Nominatim

def nearest_neighbor_heuristic(cost_matrix, demands, capacity):

  n = len(cost_matrix)
  visited = [False] * n
  tour = []
  current_node = n-1
  total_cost = 0
  load = 0

  while False in visited[:-1]:
    visited[current_node] = True
    tour.append(current_node)
    nearest = np.inf
    next_node = -1

    for j in range(n-1):
      if not visited[j] and cost_matrix[current_node][j] < nearest and current_node!=j:
        if load + demands[j] <= capacity:
          nearest = cost_matrix[current_node][j]
          next_node = j

    if next_node == -1:
        load = 0               # Return to warehouse town to change truck
        total_cost += cost_matrix[current_node][n-1]
        current_node = n-1

    else:
      total_cost += nearest
      load += demands[next_node]
      current_node = next_node

  if tour[-1] != n-1:
    total_cost += cost_matrix[current_node][n-1]
    tour.append(n-1)

  print("Trucks route: ", tour, "(where 7 is the warehouse town)")
  print("Total cost: ", total_cost, "€")

  plot_map(tour)

def plot_map(tour):
  villes = {
      0: "Breauté",
      1: "Dieppe",
      2: "Evreux",
      3: "Fécamp",
      4: "Le Havre",
      5: "Lisieux",
      6: "Rouen",
      7: "Yvetot"
  }

  geo = Nominatim(user_agent="vrp_solver")
  locs = {}

  for i, ville in villes.items():
    loc = geo.geocode(ville)
    locs[i] = (loc.latitude, loc.longitude)

  warehouse_coord = locs[len(villes)-1]
  map = folium.Map(location=warehouse_coord, zoom_start=6)

  for i in range(len(tour)):
    folium.Marker(location=locs[tour[i]], popup=villes[tour[i]]).add_to(map)
    if i > 0:
      folium.PolyLine(locations=[locs[tour[i-1]], locs[tour[i]]], color='blue').add_to(map)

  map.save('map.html')
  print("Map of tour is save in 'map.html'.")


# Execution
cost_matrix=[[0,75,115,20,35,70,70,35],[75,0,115,35,100,150,70,50],[115,115,0,120,120,75,55,90],[20,35,120,0,45,95,70,45],
       [35,100,120,45,0,60,95,60],[70,150,75,95,60,0,100,100],[70,70,55,70,95,100,0,40],[35,50,90,45,60,100,40,0]]
demands=[23,15,16,8,17,28,12]
Q=50
nearest_neighbor_heuristic(cost_matrix, demands, Q)


Trucks route:  [7, 0, 3, 1, 7, 6, 2, 4, 7, 5, 7] (where 7 is the warehouse town)
Total cost:  615 €
Map of tour is save in 'map.html'.
